# Day 1 — Selenium Edition

## Why Selenium?

The original `day1.ipynb` uses `requests` + `BeautifulSoup` which only fetches raw HTML — before JavaScript runs.
Sites like **openai.com**, **react** apps, and many modern websites render their content via JS, so they appear blank to a plain HTTP request.

Selenium launches a **real Chrome browser** headlessly, waits for JS to execute, then reads the fully-rendered page — giving us access to dynamic content.

Everything else (Gemini API, prompts, summarization) stays the same as Day 1.

In [ ]:
# This project uses uv (not pip) as the package manager
# If selenium is missing, run this from your terminal (not this cell):
#   uv add selenium webdriver-manager
print("selenium and webdriver-manager should already be installed via uv")

In [ ]:
import os
import time
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from openai import OpenAI
from IPython.display import Markdown, display

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

In [ ]:
load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')

if not api_key or not api_key.startswith("AIz"):
    print("GOOGLE_API_KEY not found or invalid — check your .env file")
else:
    print("API key found and looks good!")

In [ ]:
gemini = OpenAI(
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# Quick sanity check
response = gemini.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[{"role": "user", "content": "Say hello in one sentence."}]
)
print(response.choices[0].message.content)

## Selenium Scraper

Key differences from the `requests` approach:
- Launches a **headless Chrome** browser (invisible window)
- Waits for the page body to appear (`WebDriverWait`) so JS has time to render
- Falls back to a fixed `time.sleep(3)` for pages with no detectable load signal
- Still strips noise tags and truncates to **2,000 characters** — same limit as Day 1

In [ ]:
def make_driver():
    options = Options()
    options.add_argument("--headless=new")          # headless Chrome
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)


def fetch_website_contents(url, wait_seconds=3):
    """
    Fetch fully JS-rendered page content using Selenium.
    Returns title + body text, truncated to 2,000 characters.
    """
    driver = make_driver()
    try:
        driver.get(url)
        # Wait until <body> is present, then give JS extra time to settle
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        time.sleep(wait_seconds)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        title = soup.title.string.strip() if soup.title else "No title found"

        if soup.body:
            for tag in soup.body(["script", "style", "img", "input", "svg", "noscript"]):
                tag.decompose()
            text = soup.body.get_text(separator="\n", strip=True)
        else:
            text = ""

        return (title + "\n\n" + text)[:2_000]
    finally:
        driver.quit()

## Test the Selenium scraper

Let's try `openai.com` — a JS-heavy site that the Day 1 `requests`-based scraper could NOT handle.

In [ ]:
content = fetch_website_contents("https://openai.com")
print(content)

## Prompts

Same structure as Day 1 — system prompt sets the persona, user prompt asks for a summary of the scraped content.

In [ ]:
system_prompt = (
    "You are an assistant that analyzes the contents of a website "
    "and provides a short, accurate summary. "
    "Respond in markdown."
)

user_prompt_prefix = "Here is the content of a website. Please summarize it:\n\n"

In [ ]:
def messages_for(website_content):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website_content}
    ]

In [ ]:
def summarize(url, wait_seconds=3):
    website = fetch_website_contents(url, wait_seconds=wait_seconds)
    response = gemini.chat.completions.create(
        model="gemini-2.5-flash-lite",
        messages=messages_for(website)
    )
    return response.choices[0].message.content


def display_summary(url, wait_seconds=3):
    summary = summarize(url, wait_seconds=wait_seconds)
    display(Markdown(summary))

## Summarize complex / JS-rendered websites

These sites all **fail** with the plain `requests` scraper in Day 1.

In [ ]:
display_summary("https://openai.com")

In [ ]:
display_summary("https://anthropic.com")

In [ ]:
display_summary("https://huggingface.co")

## Notes & Limitations

- **Slower than requests** — each call launches a full Chrome process (~3–5s overhead)
- **Still blocked by some sites** — Cloudflare and aggressive anti-bot systems can detect headless browsers
- **`wait_seconds` parameter** — increase it (e.g. `display_summary(url, wait_seconds=5)`) for slower single-page apps
- **2,000 char limit** — same as Day 1; increase it if you want more context sent to the LLM (costs more tokens)